# Final paper runs

Everything that is still to run for the paper, in one queue: the zero-shot baseline, timing and invalid-output checks on the finished adapters, archived Messidor scoring, and the new training variants (learning rate and alpha), each followed by its own scoring.

1. Run **1. Setup**, then **2. What is left**. Both only read files.
2. Set `TRAINING_MODE = True` in Setup and rerun it when you want to start GPU jobs.
3. Run **4. Pilot** first. It takes about 30 minutes and exercises every kind of job on a few images.
4. Run **5. Full queue** and leave it. It works through every unfinished job in order, keeps the PC awake, records a failed job and carries on. Rerunning the cell skips finished jobs and resumes interrupted training from its last epoch.

New results are written to `D:\Afterimage_Experiments\final_runs\<tag>`, apart from the 24 verified runs. Close other GPU-heavy apps before starting: the BF16 jobs need about 14 GB of free VRAM.

[Design and reasoning](../../../docs/paper/FINAL_RUNS_DESIGN.md) · [Deliverables checklist](../../../docs/paper/DELIVERABLES_CHECKLIST.md) · [Plan](../../../docs/paper/PLAN.md)

## 1. Setup

In [ ]:
from pathlib import Path
import importlib
import sys

if Path(sys.prefix).name.casefold() != "afterimage_env" or sys.version_info[:2] != (3, 11):
    raise RuntimeError(f"Select the Python 3.11 afterimage_env kernel. Current: {sys.executable}")

sys.dont_write_bytecode = True
QWEN_DIR = Path(r"D:\Afterimage_Reproduction\finetuning\Qwen3")
HELPER_DIR = QWEN_DIR / "paper_revision"  # pinned paper_reproduction.py stays in QWEN_DIR
for directory in (QWEN_DIR, HELPER_DIR):
    if str(directory) in sys.path:
        sys.path.remove(str(directory))
    sys.path.insert(0, str(directory))
importlib.invalidate_caches()

def load_helper(name):
    module = importlib.reload(sys.modules[name]) if name in sys.modules else importlib.import_module(name)
    if Path(module.__file__).resolve() != (HELPER_DIR / f"{name}.py").resolve():
        raise ImportError(f"Expected {name} from {HELPER_DIR}, got {module.__file__}")
    return module

reproduction = load_helper("reproduce_run_02")
remaining = load_helper("remaining_runs")
final = load_helper("final_runs")

TRAINING_MODE = True  # Change to True and rerun this cell to enable GPU jobs.

SETTINGS = final.default_settings()
SETTINGS["tag"] = "final_01"        # one folder per campaign; keep it to resume or reuse results
SETTINGS["pilot_tag"] = "pilot_01"  # change it to repeat the pilot from scratch

# Stages. Turn one off to leave it out of the queue.
SETTINGS["stages"].update(
    zero_shot=True,               # S7: base model with no adapter
    benchmark=True,               # S14: time per image, and invalid outputs under text generation
    original_adapter_audit=True,  # invalid outputs of the original paper adapter (read from the C: copy)
    messidor_archived=True,       # archived 1,057-image Messidor cohort, existing adapters
    lr_study=True,                # G2/S8: learning rate, three seeds each
    alpha_study=True,             # S6: LoRA alpha, three seeds each
    extra_variants=True,          # anything listed in SETTINGS["extra_variants"]
    main_lr_grid=False,           # see section 8
    messidor_for_new_runs=True,   # score each new CLAHE-on adapter on archived Messidor
)
SETTINGS["lr_values"] = [1e-4, 2e-4]   # the finished runs used 5e-4
SETTINGS["alpha_values"] = [8, 32]     # the finished runs used 16
SETTINGS["study_rank"], SETTINGS["study_clahe"] = 16, "on"
SETTINGS["seed_pairs"] = [1, 2, 3]     # 42/3407, 43/3408, 44/3409

# Extra single runs, for example a dropout or patience trial:
# SETTINGS["extra_variants"] = [{"seed_pair": 1, "lora_dropout": 0.0},
#                               {"seed_pair": 1, "early_stopping_patience": 4}]

def launch(job, *args, **kwargs):
    if not TRAINING_MODE:
        print("Set TRAINING_MODE = True and rerun Setup, then run this cell.")
        return
    return job(*args, **kwargs)

print("Setup OK. Results folder:", final.root(SETTINGS))

Setup OK. Results folder: D:\Afterimage_Experiments\final_runs\final_01


## 2. What is left

Lists every job in the queue with its status and a time estimate. Reads saved files only.

In [2]:
jobs = final.show_plan(SETTINGS)

Final queue in D:\Afterimage_Experiments\final_runs\final_01
  [pending    ] zero_shot              zeroshot_4bit__aptos361_on  (~8 min)
  [pending    ] zero_shot              zeroshot_4bit__aptos361_off  (~8 min)
  [pending    ] zero_shot              zeroshot_bf16__aptos361_on  (~14 min)
  [pending    ] benchmark              r16_on_p1__aptos361_on  (~8 min)
  [pending    ] benchmark              r16_on_p2__aptos361_on  (~8 min)
  [pending    ] benchmark              r16_on_p3__aptos361_on  (~8 min)
  [pending    ] benchmark              r8_on_p3__aptos361_on  (~8 min)
  [pending    ] benchmark              r32_on_p1__aptos361_on  (~8 min)
  [pending    ] benchmark              bf16_r16_on_p1__aptos361_on  (~14 min)
  [pending    ] original_adapter_audit original_paper_adapter__aptos367_original  (~8 min)
  [pending    ] messidor_archived      zeroshot_4bit__messidor_archived  (~18 min)
  [pending    ] messidor_archived      r16_on_p1__messidor_archived  (~18 min)
  [pending    ] mes

## 3. Parameter counts (no GPU)

Counts the base model's parameters and each LoRA rank's trainable parameters directly from the weight files. The training logs disagree on the total (8.82 billion for 4-bit runs, 6.12 billion for BF16 runs), so this is the count to report.

In [3]:
counts = final.parameter_counts(SETTINGS)

Base model: 8,767,123,696 parameters.
  LoRA rank_8: 25,673,472 trainable (0.293% of the base), 98 MiB on disk.
  LoRA rank_16: 51,346,944 trainable (0.586% of the base), 196 MiB on disk.
  LoRA rank_32: 102,693,888 trainable (1.171% of the base), 392 MiB on disk.


## 4. Pilot

One small job of each kind: two 10-step training runs (a learning-rate variant and an alpha variant) and 12-image scoring runs for the zero-shot base, a finished 4-bit adapter, a finished BF16 adapter and archived Messidor. Check that the output ends with **Pilot passed** before starting the full queue.

In [4]:
pilot_jobs = final.show_plan(SETTINGS, pilot=True)

Pilot queue in D:\Afterimage_Experiments\final_runs\final_01__pilot_01
  [pending    ] zero_shot              zeroshot_4bit__aptos361_on  (~4 min)
  [done       ] benchmark              r16_on_p1__aptos361_on  (~4 min)
  [done       ] benchmark              bf16_r16_on_p1__aptos361_on  (~4 min)
  [pending    ] messidor_archived      zeroshot_4bit__messidor_archived  (~4 min)
  [done       ] lr_study               r16_on_lr0p0001_t42_a3407  (~6 min)
  [done       ] alpha_study            r16_on_alpha8_t42_a3407  (~6 min)
4 of 6 jobs done. Estimated GPU time left: about 0.1 h. Training times are estimates from earlier runs.


In [5]:
launch(final.run_queue, SETTINGS, pilot=True)

Set TRAINING_MODE = True and rerun Setup, then run this cell.


## 5. Full queue

Runs every unfinished job in order and writes the summary at the end. Leave it running. If it is interrupted, run this cell again: finished jobs are skipped and training resumes from its last saved epoch.

In [6]:
launch(final.run_queue, SETTINGS)

Set TRAINING_MODE = True and rerun Setup, then run this cell.


## 6. One stage at a time

Use these instead of the full queue if you want to run or repeat a single stage.

In [7]:
launch(final.run_queue, SETTINGS, only={"zero_shot"})

Set TRAINING_MODE = True and rerun Setup, then run this cell.


In [8]:
launch(final.run_queue, SETTINGS, only={"benchmark"})

Set TRAINING_MODE = True and rerun Setup, then run this cell.


In [9]:
launch(final.run_queue, SETTINGS, only={"original_adapter_audit"})

Set TRAINING_MODE = True and rerun Setup, then run this cell.


In [10]:
launch(final.run_queue, SETTINGS, only={"messidor_archived"})

Set TRAINING_MODE = True and rerun Setup, then run this cell.


In [11]:
launch(final.run_queue, SETTINGS, only={"lr_study"})

Set TRAINING_MODE = True and rerun Setup, then run this cell.


In [12]:
launch(final.run_queue, SETTINGS, only={"alpha_study"})

Set TRAINING_MODE = True and rerun Setup, then run this cell.


In [13]:
launch(final.run_queue, SETTINGS, only={"extra_variants"})

Set TRAINING_MODE = True and rerun Setup, then run this cell.


## 7. Results

Rebuilds `summary.csv` and `summary.md` from every finished job and shows the summary. Reads saved files only.

In [14]:
rows = final.summarize(SETTINGS)
summary = final.root(SETTINGS) / "summary.md"
if summary.is_file():
    from IPython.display import Markdown, display
    display(Markdown(summary.read_text(encoding="utf-8")))

No finished jobs to summarise yet.


## 8. After the learning-rate study: repeat the grid at the chosen rate

Only if you and Dr. Noor decide to adopt a different learning rate. Choose it from the **validation** losses in the summary, not from test accuracy. This repeats rank 8/16/32 with CLAHE on and off at that rate, three seeds each, skipping runs the study already produced. Expect about 20 hours.

In [15]:
# SETTINGS["main_lr"] = 1e-4
# SETTINGS["stages"]["main_lr_grid"] = True
# final.show_plan(SETTINGS)
# launch(final.run_queue, SETTINGS, only={"main_lr_grid"})

## 9. Not covered here

- **BF16 variants** (CLAHE off, other learning rates): the BF16 worker only accepts the rank 16, CLAHE-on reference. A new worker is needed.
- **RetiZero**: its own notebook, `finetuning/RetiZero/notebooks/02_retizero_runs.ipynb`.
- **Verified Messidor**: section 8 of `021_training.ipynb`, once the raw data is located. The Messidor numbers here use the archived 1,057-image export.
- **Paper tables**: these runs are summarised here. They are not yet read by `finetuning/paper_tables/build_paper_tables.py`.